# Session 4 — Evaluating Coding Agents
### Tests as Ground Truth, Trajectories, and LangSmith

*Coding Agents That Improve* · Data Science Dojo × SambaNova · **Session 4 of 6**

---

**Session 3** gave the agent an environment and watched it turn a red suite green. One task, one
run. It worked — and that tells us almost nothing.

A single green run is an anecdote. It cannot distinguish an agent that *solved* the problem from
one that got lucky, or one that quietly deleted the failing test. This session builds the thing
that can.

Coding agents have a luxury most agents don't: **ground truth**. The tests pass or they don't —
no LLM judge required for the headline number. But "tests pass" answers only the first of three
questions:

| Question | What answers it |
|---|---|
| Did it work? | The test suite — ground truth, no judge |
| Does it work *reliably*? | **pass@k** over repeated attempts |
| Did it work for the *right reasons*? | **Trajectory evaluation** + hard gates |

We build all three, wire them into **LangSmith** as tracked experiments, and finish with the
thing that makes evaluation worth the effort: a **regression gate** that catches a model swap
that makes your agent worse.

That last part is not hypothetical. It happened to us between Session 3 and today.

## 0. Setup

Three keys in `.env`: `SAMBANOVA_API_KEY`, `DAYTONA_API_KEY`, `LANGSMITH_API_KEY`.

In [1]:
# !pip install openai python-dotenv daytona langsmith pandas -q

In [2]:
import json
import os
import posixpath
import queue
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from math import comb

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from daytona import Daytona, DaytonaConfig

load_dotenv()

EXECUTOR = "gpt-oss-120b"          # the model under evaluation
CHALLENGER = "MiniMax-M3.1-Flash"  # the one we will compare it against

# max_retries matters here: an eval sweep is the most rate-limit-hungry thing
# you will run, and the SDK's default is to retry quietly - which looks exactly
# like a hang. Bound it, and let failures surface.
client  = OpenAI(base_url="https://api.sambanova.ai/v1",
                 api_key=os.environ["SAMBANOVA_API_KEY"],
                 max_retries=5, timeout=120.0)
daytona = Daytona(DaytonaConfig(api_key=os.environ["DAYTONA_API_KEY"]))

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "dsd-s4-coding-agent-eval")

print("models on SambaNova right now:")
for m in sorted(client.models.list().data, key=lambda x: x.id):
    print(f"  {m.id}")

models on SambaNova right now:


  GLM-5.3
  MiniMax-M3.1-Flash
  gemma-4-31B-it
  gpt-oss-120b


### A word on that model list

Session 3 benchmarked `MiniMax-M3` against `gpt-oss-120b` and concluded M3 was the better agent:
fewer steps, zero malformed tool calls, a tighter patch. That was a real measurement.

**`MiniMax-M3` is no longer in that list.** The lineup moved on — `MiniMax-M2.7` now returns
`410 Gone`, and M3 is unlisted. If your agent pinned a model id, it is one deprecation away from
breaking, and a silent substitution away from getting worse without telling you.

This is the case for evaluation in one paragraph. Let's build it.

---
## 1. The held-out task set

Six small bug-fix tasks. Each has a buggy source file and a suite containing **both** kinds of
test, the SWE-bench shape from Session 3:

* `FAIL_TO_PASS` — currently failing, must pass after the fix
* `PASS_TO_PASS` — currently passing, must *still* pass

Held-out matters: the agent has never seen these, and neither did we when tuning its prompt.

In [3]:
TASKS = [
  {"id": "cart-412", "file": "pricing.py",
   "brief": "apply_discount accepts out-of-range percentages.",
   "src": '''def apply_discount(price, pct):
    """Apply a percentage discount to a price, rounded to 2dp."""
    return round(price - price * pct / 100, 2)


def cart_total(items, discount_pct=0):
    """Total a cart of (unit_price, qty) pairs, then apply a discount."""
    subtotal = 0
    for price, qty in items:
        subtotal += price * qty
    return apply_discount(subtotal, discount_pct)
''',
   "test": '''import pytest
from pricing import apply_discount, cart_total

def test_basic():
    assert apply_discount(100, 10) == 90.0
    assert cart_total([(10.0, 2), (5.0, 1)]) == 25.0

def test_rejects_out_of_range():
    with pytest.raises(ValueError):
        apply_discount(100, 150)
    with pytest.raises(ValueError):
        apply_discount(100, -5)
    with pytest.raises(ValueError):
        cart_total([(10.0, 1)], 200)
'''},
  {"id": "pag-118", "file": "paging.py",
   "brief": "paginate() loses the last partial page.",
   "src": '''def paginate(items, per_page):
    """Split items into pages of per_page items each."""
    pages = []
    for i in range(0, len(items) // per_page * per_page, per_page):
        pages.append(items[i:i + per_page])
    return pages
''',
   "test": '''import pytest
from paging import paginate

def test_exact_multiple():
    assert paginate([1, 2, 3, 4], 2) == [[1, 2], [3, 4]]

def test_keeps_partial_last_page():
    assert paginate([1, 2, 3, 4, 5], 2) == [[1, 2], [3, 4], [5]]
    assert paginate([1], 10) == [[1]]

def test_empty_and_validation():
    assert paginate([], 3) == []
    with pytest.raises(ValueError):
        paginate([1, 2], 0)
'''},
  {"id": "slug-77", "file": "slug.py",
   "brief": "slugify collapses badly on punctuation.",
   "src": '''import re

def slugify(text):
    """Turn a title into a url slug."""
    return re.sub(r"\\s+", "-", text.strip().lower())
''',
   "test": '''import pytest
from slug import slugify

def test_simple():
    assert slugify("Hello World") == "hello-world"
    assert slugify("  Padded  Title  ") == "padded-title"

def test_strips_punctuation():
    assert slugify("Hello, World!") == "hello-world"
    assert slugify("C++ & Rust") == "c-rust"

def test_collapses_separators():
    assert slugify("a---b") == "a-b"
    assert slugify("a _ b") == "a-b"
    assert slugify("!!!") == ""
'''},
  {"id": "ret-203", "file": "retry.py",
   "brief": "retry() swallows the final exception and returns None.",
   "src": '''import time

def retry(fn, max_attempts=3, delay=0):
    """Call fn, retrying on exception. Returns fn's result."""
    for _ in range(max_attempts):
        try:
            return fn()
        except Exception:
            time.sleep(delay)
    return None
''',
   "test": '''import pytest
from retry import retry

def test_returns_on_success():
    assert retry(lambda: 42) == 42

def test_retries_then_succeeds():
    calls = {"n": 0}
    def flaky():
        calls["n"] += 1
        if calls["n"] < 3:
            raise RuntimeError("nope")
        return "ok"
    assert retry(flaky, max_attempts=5) == "ok"
    assert calls["n"] == 3

def test_gives_up_after_max_attempts():
    calls = {"n": 0}
    def always():
        calls["n"] += 1
        raise ValueError("boom")
    with pytest.raises(ValueError):
        retry(always, max_attempts=4)
    assert calls["n"] == 4
'''},
  {"id": "csv-55", "file": "rows.py",
   "brief": "parse_row mishandles quoted fields containing commas.",
   "src": '''def parse_row(line):
    """Parse one CSV line into a list of fields."""
    return line.rstrip("\\n").split(",")
''',
   "test": '''import pytest
from rows import parse_row

def test_plain():
    assert parse_row("a,b,c") == ["a", "b", "c"]
    assert parse_row("a,b,c\\n") == ["a", "b", "c"]

def test_quoted_fields():
    assert parse_row('a,"b,c",d') == ["a", "b,c", "d"]
    assert parse_row('"only"') == ["only"]

def test_empty_fields():
    assert parse_row("a,,c") == ["a", "", "c"]
'''},
  {"id": "ver-9", "file": "version.py",
   "brief": "bump_version cannot roll over minor/patch.",
   "src": '''def bump_version(v, part="patch"):
    """Bump a semver string like '1.2.3'."""
    major, minor, patch = (int(x) for x in v.split("."))
    if part == "patch":
        patch += 1
    elif part == "minor":
        minor += 1
    elif part == "major":
        major += 1
    return f"{major}.{minor}.{patch}"
''',
   "test": '''import pytest
from version import bump_version

def test_patch():
    assert bump_version("1.2.3") == "1.2.4"

def test_minor_resets_patch():
    assert bump_version("1.2.3", "minor") == "1.3.0"

def test_major_resets_both():
    assert bump_version("1.2.3", "major") == "2.0.0"

def test_rejects_bad_part():
    with pytest.raises(ValueError):
        bump_version("1.2.3", "epoch")
'''},
]

pd.DataFrame([{"id": t["id"], "file": t["file"], "issue": t["brief"]} for t in TASKS])

,id,file,issue
0,cart-412,pricing.py,apply_discount accepts out-of-range percentages.
1,pag-118,paging.py,paginate() loses the last partial page.
2,slug-77,slug.py,slugify collapses badly on punctuation.
3,ret-203,retry.py,retry() swallows the final exception and retur...
4,csv-55,rows.py,parse_row mishandles quoted fields containing ...
5,ver-9,version.py,bump_version cannot roll over minor/patch.


---
## 2. The harness under test

The Session 3 agent, unchanged in substance: four tools over a Daytona sandbox, a dispatcher
that turns bad calls into correctable errors, and path normalisation.

One addition that evaluation forces on you: **a timeout on the test command.**

In [4]:
REPO = "/home/daytona/project"
TEST_TIMEOUT = 45


def _rel(path):
    p = (path or ".").strip()
    if p.startswith(REPO):
        p = p[len(REPO):]
    p = p.lstrip("/") or "."
    return None if posixpath.normpath(p).startswith("..") else p


def t_list(sb, path="."):
    r = _rel(path)
    return "ERROR: path escapes repo" if r is None else sb.process.exec(f"ls -la {r}", cwd=REPO).result

def t_read(sb, path, start=None, end=None):
    r = _rel(path)
    if r is None:
        return "ERROR: path escapes repo"
    out = sb.process.exec(f"cat {r}", cwd=REPO).result or ""
    if start or end:
        L = out.splitlines()
        out = "\n".join(L[(int(start or 1) - 1):(int(end) if end else None)])
    return out

def t_write(sb, path, content):
    r = _rel(path)
    if r is None:
        return "ERROR: path escapes repo"
    sb.fs.upload_file(content.encode(), f"{REPO}/{r}")
    return f"wrote {r} ({len(content)} bytes)"

def t_tests(sb):
    # A candidate patch can loop forever. In a single demo you notice; across
    # 36 evaluation runs it silently eats your afternoon. Bound it, always.
    r = sb.process.exec(f"timeout {TEST_TIMEOUT} python3 -m pytest -q --no-header -rf", cwd=REPO)
    if r.exit_code == 124:
        return "exit_code=124\nTIMEOUT: suite did not finish — suspect an infinite loop in your change."
    return f"exit_code={r.exit_code}\n{(r.result or '')[-2000:]}"


TOOLS = [
 {"type":"function","function":{"name":"list_files","description":"List files in the repository.",
   "parameters":{"type":"object","properties":{"path":{"type":"string"}},"required":[]}}},
 {"type":"function","function":{"name":"read_file","description":"Read a file and return its exact contents.",
   "parameters":{"type":"object","properties":{"path":{"type":"string"},"start":{"type":"integer"},"end":{"type":"integer"}},"required":["path"]}}},
 {"type":"function","function":{"name":"write_file","description":"Overwrite a file with new content.",
   "parameters":{"type":"object","properties":{"path":{"type":"string"},"content":{"type":"string"}},"required":["path","content"]}}},
 {"type":"function","function":{"name":"run_tests","description":"Run the pytest suite.",
   "parameters":{"type":"object","properties":{},"required":[]}}},
]
DISPATCH = {"list_files": t_list, "read_file": t_read, "write_file": t_write, "run_tests": t_tests}
SCHEMA   = {t["function"]["name"]: set(t["function"]["parameters"]["properties"]) for t in TOOLS}


def call_tool(sb, name, args):
    if name not in DISPATCH:
        return f"ERROR: no such tool {name!r}"
    unknown = set(args) - SCHEMA.get(name, set())
    if unknown:
        return f"ERROR: unknown argument(s) {sorted(unknown)} for {name}. Accepted: {sorted(SCHEMA[name])}"
    try:
        return DISPATCH[name](sb, **args)
    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"


SYSTEM = f"""You are a coding agent working in a sandboxed git repo at {REPO}.
Fix the failing tests by editing the SOURCE code. Never edit the test file.
Read the failing test first, then the source, make the smallest correct change,
then run the tests. Repeat until everything passes. When green, reply DONE."""


def seed(sb, task):
    sb.process.exec(f"rm -rf {REPO}; mkdir -p {REPO}")
    sb.fs.upload_file(task["src"].encode(),  f"{REPO}/{task['file']}")
    sb.fs.upload_file(task["test"].encode(), f"{REPO}/test_{task['file']}")
    for c in ["git init -q -b main", "git config user.email a@b.c",
              "git config user.name A", "git add -A", "git commit -q -m base"]:
        sb.process.exec(c, cwd=REPO)


class RateLimited(Exception):
    pass


def run_episode(sb, task, model, max_steps=12):
    """One attempt at one task. Returns the outcome AND the full trajectory."""
    seed(sb, task)
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Issue {task['id']}: {task['brief']}\nThe test suite is failing. Fix it."}]
    calls, tin, tout, bad = [], 0, 0, 0
    rate_limited = False
    t0 = time.perf_counter()
    for _ in range(max_steps):
        try:
            r = client.chat.completions.create(model=model, messages=msgs, tools=TOOLS,
                                               temperature=0.2, max_tokens=4000)
        except Exception as e:
            # A rate limit is a fact about the sweep, not about the agent. Record
            # it and move on rather than letting it masquerade as a failed task.
            if "rate_limit" in str(e).lower() or "429" in str(e):
                rate_limited = True
                break
            raise
        tin += r.usage.prompt_tokens; tout += r.usage.completion_tokens
        m = r.choices[0].message
        msgs.append(m.model_dump(exclude_none=True))
        if not m.tool_calls:
            break
        for tc in m.tool_calls:
            a = json.loads(tc.function.arguments or "{}")
            out = call_tool(sb, tc.function.name, a)
            if str(out).startswith(("ERROR: unknown", "ERROR: no such")):
                bad += 1
            calls.append({"tool": tc.function.name, "args": a})
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(out)[:6000]})
    wall = time.perf_counter() - t0
    final   = sb.process.exec(f"timeout {TEST_TIMEOUT} python3 -m pytest -q --no-header", cwd=REPO)
    changed = sb.process.exec("git diff --name-only", cwd=REPO).result.strip().splitlines()
    diff    = sb.process.exec("git diff", cwd=REPO).result or ""
    return {"task": task["id"], "model": model, "solved": final.exit_code == 0,
            "rate_limited": rate_limited,
            "calls": calls, "steps": len(calls), "wall": wall,
            "in": tin, "out": tout, "bad_tool_calls": bad,
            "changed": changed, "diff": diff}

### A pool of sandboxes

Evaluation is embarrassingly parallel and the agent spends most of its time waiting on the
model. One sandbox per worker, reused across tasks — creating them is cheap, but not free.

In [5]:
class Pool:
    """A fixed set of sandboxes, handed out one per worker thread."""

    def __init__(self, n):
        self.q = queue.Queue(); self.boxes = []
        for _ in range(n):
            sb = daytona.create(); self.boxes.append(sb); self.q.put(sb)

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for sb in self.boxes:
            try:
                sb.delete()
            except Exception:
                pass   # never let cleanup mask the real error

    def run(self, fn, jobs, label=""):
        """Map fn over jobs, one sandbox per worker.

        Use a real executor. An earlier version of this throttled with a
        hand-rolled `while sum(t.is_alive() ...)` loop; it pinned a core at
        100% and wedged the run. Concurrency primitives are a solved problem -
        do not hand-roll them.
        """
        done = [0]
        lock = threading.Lock()

        def work(job):
            sb = self.q.get()
            try:
                r = fn(sb, job)
            finally:
                self.q.put(sb)
            with lock:
                done[0] += 1
                print(f"\r  {label}{done[0]}/{len(jobs)} episodes", end="", flush=True)
            return r

        with ThreadPoolExecutor(max_workers=len(self.boxes)) as ex:
            out = list(ex.map(work, jobs))
        print()
        return out

---
## 3. Question 1 — did it work?

The easy one, and the reason coding agents are a gift to evaluate: run the suite.

In [6]:
K = 3          # attempts per task, for pass@k
WORKERS = 2    # one sandbox each. Raise it and you WILL hit a 429.

jobs = [(t, EXECUTOR) for t in TASKS for _ in range(K)]
print(f"{len(TASKS)} tasks × {K} attempts = {len(jobs)} episodes on {EXECUTOR}\n")

t0 = time.perf_counter()
with Pool(WORKERS) as pool:
    episodes = pool.run(lambda sb, j: run_episode(sb, j[0], j[1]), jobs, label="")
elapsed = time.perf_counter() - t0

solved = sum(e["solved"] for e in episodes)
print(f"{solved}/{len(episodes)} episodes solved, in {elapsed:.0f}s wall-clock")

6 tasks × 3 attempts = 18 episodes on gpt-oss-120b



  1/18 episodes

  2/18 episodes

  3/18 episodes

  4/18 episodes

  5/18 episodes

... (10 outputs left out)

  16/18 episodes

  17/18 episodes

  18/18 episodes

14/18 episodes solved, in 124s wall-clock


### The constraint nobody budgets for

A single agent run is a handful of model calls. An evaluation sweep is tasks × attempts × models
× *steps per episode* — and it is the most rate-limit-hungry thing you will ever point at an API.

Building this session, three concurrent workers tripped a `429` seven episodes in. The failure is
nastier than it sounds: the SDK retries with backoff by default, so a rate limit does not look
like an error, it looks like a **hang**. Bound your retries, keep concurrency modest, and treat a
429 as a fact about the sweep rather than a failed task.

---
## 4. Question 2 — does it work *reliably*?

Per-task success rate is the number that matters, and a single run cannot produce it. `pass@k`
is the probability that **at least one of k** attempts succeeds — the same unbiased estimator
from Session 2 (Chen et al., 2021):

$$\text{pass@}k = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$

In [7]:
def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1.0 - comb(n - c, k) / comb(n, k)


by_task = {}
for e in episodes:
    by_task.setdefault(e["task"], []).append(e)

outcome = pd.DataFrame([
    {"task": tid,
     "solved": f"{sum(x['solved'] for x in eps)}/{len(eps)}",
     "pass@1": round(pass_at_k(len(eps), sum(x["solved"] for x in eps), 1), 2),
     f"pass@{K}": round(pass_at_k(len(eps), sum(x["solved"] for x in eps), K), 2),
     "median steps": int(pd.Series([x["steps"] for x in eps]).median()),
     "median s": round(pd.Series([x["wall"] for x in eps]).median(), 1)}
    for tid, eps in sorted(by_task.items())
])
overall_p1 = round(sum(e["solved"] for e in episodes) / len(episodes), 3)
print(f"Overall pass@1 across the task set: {overall_p1}\n")
outcome

Overall pass@1 across the task set: 0.778



,task,solved,pass@1,pass@3,median steps,median s
0,cart-412,3/3,1.00,1.0,8,11.7
1,csv-55,3/3,1.00,1.0,9,16.9
2,pag-118,3/3,1.00,1.0,9,4.5
3,ret-203,3/3,1.00,1.0,11,20.9
4,slug-77,1/3,0.33,1.0,9,17.2
5,ver-9,1/3,0.33,1.0,4,15.8


### If every row says 3/3, your task set is the problem

A benchmark everybody passes measures nothing. If `pass@1` comes out at 1.00 across the board,
you have not proven the agent is good — you have proven the tasks are too easy to discriminate,
and the suite can no longer tell an improvement from a regression.

The fix is not a better metric, it is harder tasks: longer files, more than one call site to
change, bugs whose failing test does not point straight at the fault. Keep a few easy ones as a
smoke test, but an eval set that cannot fail cannot inform.

Two things to read off that table. First, **the per-task spread**: an agent that is excellent on
four tasks and hopeless on two has a very different failure profile from one that is mediocre
everywhere, and a single aggregate number hides which you have.

Second, where `pass@1` and `pass@{K}` diverge, the agent *can* solve the task but does not do so
dependably. That gap is the thing best-of-N buys back — and the thing an improvement loop should
be closing.

---
## 5. Question 3 — did it work for the *right reasons*?

Now the part the test suite cannot tell you. Two runs can both end green while one read the
failing test, made a surgical change and verified it, and the other thrashed for eleven steps
and got there by luck.

Worse: a run can end green by **deleting the failing test**.

So we split this into two different kinds of check, and the distinction matters more than the
metrics themselves.

In [8]:
def trajectory(ep, task):
    """Rule-based facts about HOW the agent worked. No judge, no model call."""
    names = [c["tool"] for c in ep["calls"]]
    test_file = f"test_{task['file']}"
    reads = [c["args"].get("path", "") for c in ep["calls"] if c["tool"] == "read_file"]
    first_write = next((i for i, n in enumerate(names) if n == "write_file"), None)

    read_test_first = (any(test_file in (c["args"].get("path") or "")
                           for c in ep["calls"][:first_write] if c["tool"] == "read_file")
                       if first_write is not None else False)
    checks = [i for i, n in enumerate(names) if n == "run_tests"]
    verified = bool(checks) and (first_write is None or checks[-1] > first_write)

    sigs = [f"{c['tool']}:{json.dumps(c['args'], sort_keys=True)}" for c in ep["calls"]]
    run = best = 1 if sigs else 0
    for i in range(1, len(sigs)):
        run = run + 1 if sigs[i] == sigs[i-1] else 1
        best = max(best, run)

    READONLY = {"list_files", "read_file"}
    r = spin = 0
    for n in names:
        r = r + 1 if n in READONLY else 0
        spin = max(spin, r)

    return {"read_test_before_edit": read_test_first,
            "verified_before_stop": verified,
            "edited_tests": any("test_" in f for f in ep["changed"]),
            "repeat_streak": best,
            "readonly_streak": spin,
            "redundant_reads": sum(1 for i in range(1, len(reads)) if reads[i] == reads[i-1]),
            "bad_tool_calls": ep["bad_tool_calls"],
            "steps": ep["steps"]}


def _decay(value, free, rate=0.25):
    return 1.0 if value <= free else max(0.0, 1 - rate * (value - free))


def trajectory_score(t):
    """0–1 EFFICIENCY. Did it take a sane path, regardless of whether it arrived?"""
    parts = [
        1.0 if t["read_test_before_edit"] else 0.0,
        _decay(t["readonly_streak"], free=3),        # spinning without acting
        _decay(t["repeat_streak"], free=1),          # identical call, repeated
        _decay(t["bad_tool_calls"], free=0, rate=0.34),
        _decay(t["steps"], free=10, rate=0.05),      # verbosity
    ]
    return round(sum(parts) / len(parts), 3)


def gates(t):
    """Hard pass/fail. Any False invalidates the run, whatever else it scored."""
    return {"did_not_edit_tests": not t["edited_tests"],
            "verified_before_stopping": t["verified_before_stop"]}

### Why gates are not just low-weighted score components

It is tempting to roll everything into one number. Here is why that fails — three real
trajectories, scored:

In [9]:
def _synth(reads, seq, changed):
    calls, ri = [], 0
    for t in seq:
        if t in ("read_file", "list_files"):
            calls.append({"tool": t, "args": {"path": reads[min(ri, len(reads)-1)]}}); ri += 1
        else:
            calls.append({"tool": t, "args": {}})
    return {"calls": calls, "steps": len(seq), "bad_tool_calls": 0, "changed": list(changed)}

demo_task = {"file": "version.py"}
cases = [
    ("spun its wheels",
     ["version.py", "test_version.py", "version.py", "x", "x", "x", "x", "x"],
     "list_files run_tests read_file read_file write_file run_tests list_files list_files list_files list_files list_files".split(),
     ("version.py",)),
    ("clean run",
     ["version.py", "test_version.py", "version.py"],
     "run_tests list_files read_file read_file write_file run_tests run_tests write_file run_tests".split(),
     ("version.py",)),
    ("deleted the failing test",
     ["version.py", "test_version.py"],
     "read_file read_file write_file run_tests".split(),
     ("test_version.py",)),
]
rows = []
for label, reads, seq, changed in cases:
    tr = trajectory(_synth(reads, seq, changed), demo_task)
    rows.append({"trajectory": label, "efficiency": trajectory_score(tr), **gates(tr)})
pd.DataFrame(rows)

,trajectory,efficiency,did_not_edit_tests,verified_before_stopping
0,spun its wheels,0.69,True,True
1,clean run,0.95,True,True
2,deleted the failing test,1.00,False,True


The cheating run earns a **perfect efficiency score**. It was direct, it did not thrash, it
verified its work — it simply solved a different problem from the one we set.

No weighting fixes that. An average lets a cheat trade a gate against efficiency, and the whole
point of a gate is that it cannot be traded. **Gates are boolean and they are veto.**

### Scoring the real runs

In [10]:
task_by_id = {t["id"]: t for t in TASKS}
scored = []
for e in episodes:
    tr = trajectory(e, task_by_id[e["task"]])
    g  = gates(tr)
    scored.append({**e, "traj": tr, "efficiency": trajectory_score(tr),
                   "gates_pass": all(g.values()), **g})

traj_df = pd.DataFrame([
    {"task": tid,
     "solved": f"{sum(x['solved'] for x in eps)}/{len(eps)}",
     "efficiency": round(pd.Series([x["efficiency"] for x in eps]).mean(), 2),
     "max spin": max(x["traj"]["readonly_streak"] for x in eps),
     "read test first": f"{sum(x['traj']['read_test_before_edit'] for x in eps)}/{len(eps)}",
     "gates pass": f"{sum(x['gates_pass'] for x in eps)}/{len(eps)}"}
    for tid, eps in sorted(
        {t: [s for s in scored if s["task"] == t] for t in by_task}.items())
])
traj_df

,task,solved,efficiency,max spin,read test first,gates pass
0,cart-412,3/3,1.00,3,3/3,3/3
1,csv-55,3/3,0.94,3,3/3,3/3
2,pag-118,3/3,0.96,3,3/3,3/3
3,ret-203,3/3,0.84,4,3/3,3/3
4,slug-77,1/3,0.95,3,3/3,2/3
5,ver-9,1/3,0.87,3,1/3,3/3


---
## 6. A judge for the thing rules cannot see

Rules caught cheating and thrashing. They cannot tell you whether the *patch* is any good — is
it minimal, does it touch unrelated code, would you approve it in review?

That is a judgement call, so we use a model — but we keep it narrow. The judge never decides
whether the task was solved; the tests already did. It only rates the diff.

In [11]:
JUDGE_MODEL = "gpt-oss-120b"
JUDGE_PROMPT = """You are reviewing a patch produced by a coding agent.

ISSUE: {brief}

PATCH:
{diff}

Rate the patch 1-5 on each criterion and reply with ONLY a JSON object:
{{"minimal": <1-5>, "targeted": <1-5>, "would_approve": true|false, "reason": "<one sentence>"}}

minimal      5 = smallest change that fixes it; 1 = sprawling rewrite
targeted     5 = touches only what the issue requires; 1 = unrelated edits
would_approve  would you approve this in code review?"""


def judge_diff(brief, diff):
    if not diff.strip():
        return {"minimal": 0, "targeted": 0, "would_approve": False, "reason": "empty diff"}
    r = client.chat.completions.create(
        model=JUDGE_MODEL, temperature=0,
        max_tokens=1200,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(brief=brief, diff=diff[:4000])}])
    txt = (r.choices[0].message.content or "").strip()
    m = __import__("re").search(r"\{.*\}", txt, __import__("re").S)
    try:
        return json.loads(m.group(0)) if m else {"minimal": 0, "targeted": 0,
                                                 "would_approve": False, "reason": "unparseable"}
    except Exception:
        return {"minimal": 0, "targeted": 0, "would_approve": False, "reason": "unparseable"}


# judge one solved run per task — the judge is the expensive evaluator, so spend it sparingly
judged = []
for tid, eps in sorted(by_task.items()):
    win = next((e for e in eps if e["solved"]), eps[0])
    v = judge_diff(task_by_id[tid]["brief"], win["diff"])
    judged.append({"task": tid, "solved": win["solved"], **{k: v.get(k) for k in
                   ("minimal", "targeted", "would_approve")}, "reason": (v.get("reason") or "")[:60]})
pd.DataFrame(judged)

,task,solved,minimal,targeted,would_approve,reason
0,cart-412,True,5,5,True,Adds proper validation for discount percent wi...
1,csv-55,True,5,5,True,The patch replaces the naive split with csv.re...
2,pag-118,True,5,5,True,Fixes the missing partial page by iterating ov...
3,ret-203,True,5,5,True,The patch directly fixes the swallowed excepti...
4,slug-77,True,4,5,True,Fixes punctuation handling by collapsing non‑a...
5,ver-9,True,4,5,True,Fixes rollover correctly and adds useful valid...


---
## 7. LangSmith — making it a tracked experiment

Everything so far lives in this notebook's memory. That is fine once and useless over time: the
question evaluation exists to answer is *"is this version better than the last one?"*, and that
needs results you can store, compare and click into.

We push the task set up as a **dataset**, then run the agent as a tracked **experiment** against
it, with our evaluators attached.

In [12]:
from langsmith import Client

ls = Client()
DATASET = "dsd-s4-coding-agent-tasks"

existing = {d.name for d in ls.list_datasets()}
if DATASET in existing:
    ds = ls.read_dataset(dataset_name=DATASET)
    print(f"reusing dataset '{DATASET}'")
else:
    ds = ls.create_dataset(dataset_name=DATASET,
                           description="Held-out bug-fix tasks for the DSD coding agent")
    ls.create_examples(dataset_id=ds.id, examples=[
        {"inputs": {"task_id": t["id"], "brief": t["brief"], "file": t["file"]},
         "outputs": {"must_pass": True}} for t in TASKS])
    print(f"created dataset '{DATASET}' with {len(TASKS)} examples")

print(f"  {len(list(ls.list_examples(dataset_id=ds.id)))} examples")

reusing dataset 'dsd-s4-coding-agent-tasks'
  6 examples


The target function is the thing under test. The evaluators are the three questions, now as
code LangSmith will run for us and store against every example.

In [13]:
from langsmith import evaluate

_eval_pool = Pool(WORKERS)

def make_target(model):
    def target(inputs):
        task = task_by_id[inputs["task_id"]]
        sb = _eval_pool.q.get()
        try:
            ep = run_episode(sb, task, model)
        finally:
            _eval_pool.q.put(sb)
        tr = trajectory(ep, task)
        return {"solved": ep["solved"], "steps": ep["steps"],
                "efficiency": trajectory_score(tr), **gates(tr),
                "diff": ep["diff"][:2000]}
    return target


def ev_solved(outputs, reference_outputs):
    return {"key": "tests_pass", "score": bool(outputs.get("solved"))}

def ev_efficiency(outputs, reference_outputs):
    return {"key": "trajectory_efficiency", "score": float(outputs.get("efficiency", 0))}

def ev_gates(outputs, reference_outputs):
    ok = bool(outputs.get("did_not_edit_tests")) and bool(outputs.get("verified_before_stopping"))
    return {"key": "gates_pass", "score": ok}

EVALUATORS = [ev_solved, ev_efficiency, ev_gates]

In [14]:
results = {}
for model in (EXECUTOR, CHALLENGER):
    print(f"\n--- experiment: {model} ---")
    res = evaluate(make_target(model), data=DATASET, evaluators=EVALUATORS,
                   experiment_prefix=f"s4-{model}", max_concurrency=WORKERS,
                   metadata={"model": model, "session": "s4"})
    rows = list(res)
    agg = {}
    for r in rows:
        for e in r["evaluation_results"]["results"]:
            agg.setdefault(e.key, []).append(float(e.score or 0))
    results[model] = {k: round(sum(v) / len(v), 3) for k, v in agg.items()}
    print("  ", results[model])

_eval_pool.__exit__()


--- experiment: gpt-oss-120b ---


View the evaluation results for experiment: 's4-gpt-oss-120b-5dbcc86f' at:
https://smith.langchain.com/o/cfd65fb2-7bd6-55cd-a050-8e6aac6c3b8a/datasets/8f588947-42f1-4d07-872f-079a7444f5f4/compare?selectedSessions=c9f49777-0004-4968-8b2f-d5cb93021fa8




0it [00:00, ?it/s]

   {'tests_pass': 1.0, 'trajectory_efficiency': 0.927, 'gates_pass': 1.0}

--- experiment: MiniMax-M3.1-Flash ---


View the evaluation results for experiment: 's4-MiniMax-M3.1-Flash-ec44e0e2' at:
https://smith.langchain.com/o/cfd65fb2-7bd6-55cd-a050-8e6aac6c3b8a/datasets/8f588947-42f1-4d07-872f-079a7444f5f4/compare?selectedSessions=d327f601-c0e5-4899-a77f-bef9a5f923b2




0it [00:00, ?it/s]

   {'tests_pass': 0.667, 'trajectory_efficiency': 0.802, 'gates_pass': 1.0}


In [15]:
board = pd.DataFrame(results).T
board.index.name = "model"
board

,tests_pass,trajectory_efficiency,gates_pass
model,,,
gpt-oss-120b,1.000,0.927,1.0
MiniMax-M3.1-Flash,0.667,0.802,1.0


Those experiments are now in LangSmith side by side: same dataset, same evaluators, per-example
scores, and a full trace for every run. When one regresses you click the row and watch the agent
do it.

---
## 8. The regression gate

A scorecard you look at is a report. A scorecard that can **fail a build** is a control.

This is where the session started: Session 3 measured `MiniMax-M3` as the better executor. The
model list moved on, and the obvious substitution — the similarly-named `MiniMax-M3.1-Flash` —
is not obviously an upgrade for *this* agent. A gate is what turns that from a surprise in
production into a failed check.

In [16]:
# A real gate does not compare against numbers somebody typed once. It compares
# against the LAST KNOWN GOOD run - the scorecard of whatever is in production
# today, committed alongside the code.
BASELINE = {k: round(v, 3) for k, v in results[EXECUTOR].items()}
TOLERANCE = 0.05

print(f"baseline = {EXECUTOR} (last known good): {BASELINE}\n")


def regression_gate(scorecard, baseline=BASELINE, tol=TOLERANCE):
    failures = []
    for metric, floor in baseline.items():
        got = scorecard.get(metric, 0.0)
        if metric == "gates_pass":
            if got < floor:                       # gates never get a tolerance
                failures.append(f"{metric}: {got:.2f} < {floor:.2f} (no tolerance on gates)")
        elif got < floor - tol:
            failures.append(f"{metric}: {got:.3f} < {floor:.3f} - {tol:.2f}")
    return (not failures), failures


for model, card in results.items():
    ok, failures = regression_gate(card)
    tag = "baseline" if model == EXECUTOR else "candidate"
    print(f"{model:22} ({tag:9}) {'PASS' if ok else 'FAIL'}")
    for f in failures:
        print(f"     {f}")

baseline = gpt-oss-120b (last known good): {'tests_pass': 1.0, 'trajectory_efficiency': 0.927, 'gates_pass': 1.0}

gpt-oss-120b           (baseline ) PASS
MiniMax-M3.1-Flash     (candidate) FAIL
     tests_pass: 0.667 < 1.000 - 0.05
     trajectory_efficiency: 0.802 < 0.927 - 0.05


Note the asymmetry: `tests_pass` and `efficiency` get a tolerance band because they are noisy —
re-run this notebook and they will move. `gates_pass` gets none. An agent that edits the test
file has not regressed slightly; it has stopped being an agent you can trust.

And note what just happened. Nobody chose a threshold to make the candidate look bad; the
baseline *is* the incumbent's own measured scorecard. The swap that looked like a routine model
upgrade — the similarly-named successor to the model Session 3 recommended — does not clear the
bar the current model set. That is the whole job of a gate: it is the difference between finding
out now and finding out from your users.

---
## Takeaways

1. **A single green run is an anecdote.** It cannot separate solved from lucky from cheated.
2. **Tests are ground truth — for the first question only.** They say *whether* it worked, never
   whether it worked reliably or for the right reasons.
3. **pass@k is the reliability number.** The gap between `pass@1` and `pass@k` is exactly what an
   improvement loop should be closing.
4. **Trajectory evaluation catches what outcomes hide** — spinning, thrashing, and the agent that
   went green by deleting your test.
5. **Gates are veto, not weights.** The cheating run scored perfect efficiency. Any metric a
   cheat can average away is not protecting you.
6. **Use a judge only where rules cannot reach** — patch quality, not correctness. Never let the
   judge decide what the tests already know.
7. **An experiment you cannot compare is a number you cannot act on.** Datasets plus tracked
   experiments are what make "better than last week" a question with an answer.

### Next session
**Session 5 — Improving Your Agent From Its Traces.** We have the measurements and the failing
runs. Now we read them, diagnose the recurring failure modes, and update the agent — then A/B it
against this baseline and watch these numbers move.